# Predicting Student Health Risk - Baseline Modeling

This public notebook builds a leaderboard-ready baseline for **Playground Series S6E7** and writes a notebook-generated `submission.csv`.

This version keeps the v8 **balanced LGBM/XGB domain ensemble** champion, then tests a **synthetic-geometry feature forge** inspired by strong public signal-engine notebooks: joint ranks, quantile bins, group-median deviations, and rank composites on top of the domain base.

Previously rejected blend/stack/threshold ablations stay disabled so runtime goes to the geometry LGBM/XGB experiment.

Promotion still requires `>= 0.0002` balanced-accuracy gain and no macro-F1 loss versus v8, then a public score above `0.94959`.


## 1. Setup

We optimize for reliable public-notebook workflow:

- **row-safe features only**: every geometry feature is computed independently for train and test rows;
- **joint train+test geometry**: ranks, quantile bins, and frequency encodings use the combined table without target leakage;
- **domain base preserved**: geometry augments, not replaces, the existing domain-ordered lifestyle features;
- **minority-class guardrails**: local accuracy alone is not trusted after the failed unweighted HGB submission;
- **one final artifact**: the champion gate writes one `/kaggle/working/submission.csv`.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import warnings
import zipfile

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

SEED = 42
N_SPLITS = 3
EPS = 1e-6

# v19: synthetic-geometry feature forge after v18 stacking failed the gate.
RUN_EXPLORATORY_ABLATIONS = False
RUN_HP_SEARCH = False
RUN_MULTI_SEED = False
RUN_FIVE_FOLD = False
RUN_CATBOOST_DIVERSITY = False
RUN_CROSSFIT_THRESHOLDS = False
RUN_STACKING = False
RUN_GEOMETRY_FORGE = True
MULTI_SEEDS = [42, 0, 7, 17, 99]

try:
    import subprocess as _sp
    GPU_AVAILABLE = False
    _smi = _sp.run(['nvidia-smi'], capture_output=True, text=True)
    if _smi.returncode == 0:
        GPU_AVAILABLE = True
except Exception:
    GPU_AVAILABLE = False
print('GPU available:', GPU_AVAILABLE)

REQUIRED_FILES = ['train.csv', 'test.csv', 'sample_submission.csv']


def has_required_files(candidate: Path) -> bool:
    return all((candidate / name).exists() for name in REQUIRED_FILES)


def download_competition_data(target: Path):
    target.mkdir(parents=True, exist_ok=True)
    if has_required_files(target):
        return target

    try:
        subprocess.run(
            ['kaggle', 'competitions', 'download', '-c', 'playground-series-s6e7', '-p', str(target)],
            check=True,
        )
        for zip_path in target.glob('*.zip'):
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(target)
    except Exception as exc:
        print('Competition-data download fallback failed:', type(exc).__name__, exc)

    return target if has_required_files(target) else None


def find_data_dir() -> Path:
    candidates = [
        Path('/kaggle/input/playground-series-s6e7'),
        Path('../input/playground-series-s6e7'),
        Path('data'),
    ]
    for candidate in candidates:
        if has_required_files(candidate):
            return candidate

    input_root = Path('/kaggle/input')
    if input_root.exists():
        for train_path in input_root.rglob('train.csv'):
            candidate = train_path.parent
            if has_required_files(candidate):
                return candidate

    if Path('/kaggle/working').exists():
        downloaded = download_competition_data(Path('/kaggle/working/data'))
        if downloaded is not None:
            return downloaded

    available = []
    if input_root.exists():
        available = sorted(str(p) for p in input_root.glob('*'))[:30]
    raise FileNotFoundError(
        'Could not find competition CSV files. Attach playground-series-s6e7, '
        f'enable Kaggle API fallback, or provide local data/. Available /kaggle/input entries: {available}'
    )


DATA_DIR = find_data_dir()

WORK_DIR = Path('/kaggle/working')
if not WORK_DIR.exists():
    WORK_DIR = Path('.')

print('Data directory:', DATA_DIR.resolve())
print('Work directory:', WORK_DIR.resolve())

## 2. Load Data

The competition target is **`health_condition`**. We keep `id` only for the final submission and never use it as a predictive feature.

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET = 'health_condition'
ID_COL = 'id'

feature_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]
X_raw = train[feature_cols].copy()
y = train[TARGET].copy()
X_test_raw = test[feature_cols].copy()

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Target distribution:')
display(y.value_counts(normalize=True).rename('share').to_frame().style.format({'share': '{:.2%}'}))

## 3. Domain-Ordered Feature Engineering

The strongest EDA signal is not a single raw column; it is the combination of **sleep**, **activity**, **stress**, **BMI**, and **lifestyle choices**.

For tree models that do not natively understand ordered categories, we add compact ordinal versions where the order has health meaning. These are not target encodings; they are **domain-ordered encodings** computed from row values only.

In [ ]:
ORDERED_MAPS = {
    'diet_type': {'veg': 0.0, 'non-veg': 1.0, 'balanced': 2.0},
    'stress_level': {'high': 0.0, 'medium': 1.0, 'low': 2.0},
    'sleep_quality': {'poor': 0.0, 'average': 1.0, 'good': 2.0},
    'physical_activity_level': {'sedentary': 0.0, 'moderate': 1.0, 'active': 2.0},
    'smoking_alcohol': {'yes': 0.0, 'occasional': 1.0, 'no': 2.0},
}


def _safe_num(df: pd.DataFrame, col: str, default=np.nan) -> pd.Series:
    if col in df.columns:
        return pd.to_numeric(df[col], errors='coerce')
    return pd.Series(default, index=df.index, dtype='float64')


def build_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    numeric_cols = out.select_dtypes(include=[np.number]).columns.tolist()
    out['missing_count'] = out.isna().sum(axis=1)
    out['numeric_missing_count'] = out[numeric_cols].isna().sum(axis=1) if numeric_cols else 0

    ordinal_cols = []
    for col, mapping in ORDERED_MAPS.items():
        if col in out.columns:
            new_col = f'{col}_ord'
            out[new_col] = out[col].map(mapping).astype('float64')
            ordinal_cols.append(new_col)

    sleep = _safe_num(out, 'sleep_duration')
    steps = _safe_num(out, 'step_count')
    exercise = _safe_num(out, 'exercise_duration')
    bmi = _safe_num(out, 'bmi')
    calories = _safe_num(out, 'calorie_expenditure')
    heart_rate = _safe_num(out, 'heart_rate')

    sleep_quality_ord = out.get('sleep_quality_ord', pd.Series(np.nan, index=out.index))
    activity_ord = out.get('physical_activity_level_ord', pd.Series(np.nan, index=out.index))
    stress_ord = out.get('stress_level_ord', pd.Series(np.nan, index=out.index))

    if ordinal_cols:
        out['lifestyle_balance_score'] = out[ordinal_cols].mean(axis=1)
        out['lifestyle_low_signal_count'] = (out[ordinal_cols] <= 0.5).sum(axis=1)
        out['lifestyle_high_signal_count'] = (out[ordinal_cols] >= 1.5).sum(axis=1)

    out['sleep_recovery_score'] = np.exp(-((sleep - 8.0) ** 2) / 6.0) * (1.0 + 0.20 * sleep_quality_ord)
    out['activity_volume_score'] = np.log1p(steps.clip(lower=0)) * (1.0 + 0.20 * activity_ord) + exercise / 45.0
    out['stress_sleep_pressure'] = (2.0 - stress_ord) * (8.0 - sleep).clip(lower=0)
    out['bmi_activity_ratio'] = bmi / (np.log1p(steps.clip(lower=0)) + 1.0)
    out['calorie_per_bmi'] = calories / (bmi + EPS)
    out['steps_per_bmi'] = steps / (bmi + EPS)
    out['activity_intensity'] = steps / (exercise + 1.0)
    out['calorie_per_step'] = calories / (steps + 1.0)
    out['exercise_per_sleep'] = exercise / (sleep + EPS)
    out['steps_per_sleep'] = steps / (sleep + EPS)
    out['bmi_sleep_interaction'] = bmi * sleep

    out['low_sleep_flag'] = (sleep < 6.0).astype('float64')
    out['long_sleep_flag'] = (sleep > 9.0).astype('float64')
    out['bmi_risk_flag'] = ((bmi < 18.5) | (bmi >= 25.0)).astype('float64')
    out['cardio_rate_flag'] = ((heart_rate < 60.0) | (heart_rate > 100.0)).astype('float64')
    out['high_stress_low_sleep_flag'] = ((stress_ord <= 0.5) & (sleep < 6.5)).astype('float64')
    out['active_good_sleep_flag'] = ((activity_ord >= 1.5) & (sleep_quality_ord >= 1.5)).astype('float64')

    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def add_targeted_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """Row-safe cross features that amplify the EDA lifestyle combinations."""
    out = df.copy()

    sleep = _safe_num(out, 'sleep_duration')
    steps = _safe_num(out, 'step_count')
    exercise = _safe_num(out, 'exercise_duration')
    bmi = _safe_num(out, 'bmi')
    calories = _safe_num(out, 'calorie_expenditure')
    sleep_quality_ord = out.get('sleep_quality_ord', pd.Series(np.nan, index=out.index))
    activity_ord = out.get('physical_activity_level_ord', pd.Series(np.nan, index=out.index))
    stress_ord = out.get('stress_level_ord', pd.Series(np.nan, index=out.index))

    # Explicit names from the docs strategy note.
    out['sleep_activity_score'] = (
        (sleep / 8.0) * (1.0 + 0.25 * activity_ord)
        + np.log1p(steps.clip(lower=0)) / 10.0
        + exercise / 60.0
    )
    out['stress_sleep_risk'] = (
        (2.0 - stress_ord)
        * ((8.0 - sleep).clip(lower=0) / 8.0 + (2.0 - sleep_quality_ord) / 2.0)
    )
    out['stress_activity_mismatch'] = (2.0 - stress_ord) * (2.0 - activity_ord)
    out['bmi_stress_interaction'] = bmi * (2.0 - stress_ord)
    out['calorie_activity_gap'] = calories - (
        0.04 * steps.clip(lower=0) + 6.0 * exercise.clip(lower=0)
    )
    out['recovery_deficit'] = ((8.0 - sleep).clip(lower=0)) * (2.0 - sleep_quality_ord)

    for col, name in [
        ('sleep_duration', 'missing_sleep_flag'),
        ('step_count', 'missing_steps_flag'),
        ('exercise_duration', 'missing_exercise_flag'),
        ('bmi', 'missing_bmi_flag'),
        ('calorie_expenditure', 'missing_calories_flag'),
    ]:
        if col in out.columns:
            out[name] = out[col].isna().astype('float64')
        else:
            out[name] = 0.0

    out = out.replace([np.inf, -np.inf], np.nan)
    return out



def build_geometry_forge(train_raw: pd.DataFrame, test_raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Target-free synthetic geometry: ranks, q-bins, group deviations, and freq encodings."""
    feature_cols = [c for c in train_raw.columns if c not in [ID_COL]]
    tr = train_raw[feature_cols].copy()
    te = test_raw[feature_cols].copy()
    combined = pd.concat([tr, te], axis=0, ignore_index=True)
    n_train = len(tr)

    num_cols = combined.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in feature_cols if c not in num_cols]

    geom = pd.DataFrame(index=combined.index)
    for col in num_cols:
        rank = combined[col].rank(pct=True, method='average')
        geom[f'{col}__rank'] = rank.astype('float64')
        geom[f'{col}__q32'] = np.floor(rank * 32).clip(0, 31).astype('float64')

    for col in cat_cols:
        combined[col] = combined[col].fillna('__missing__').astype(str).str.strip().str.lower()

    key_groups = [
        c for c in ['stress_level', 'sleep_quality', 'physical_activity_level', 'diet_type', 'smoking_alcohol', 'gender']
        if c in cat_cols
    ]
    key_nums = [
        c for c in ['sleep_duration', 'bmi', 'step_count', 'exercise_duration', 'calorie_expenditure', 'heart_rate', 'water_intake']
        if c in num_cols
    ]
    for group in key_groups:
        for num in key_nums:
            med = combined.groupby(group, observed=True)[num].transform('median')
            geom[f'{num}__dev__{group}'] = (combined[num] - med).astype('float64')

    for col in cat_cols:
        freq = combined[col].value_counts(normalize=True)
        geom[f'{col}__freq'] = combined[col].map(freq).astype('float64')

    def _rank(name: str) -> pd.Series | None:
        col = f'{name}__rank'
        return geom[col] if col in geom.columns else None

    sleep_rank = _rank('sleep_duration')
    step_rank = _rank('step_count')
    exercise_rank = _rank('exercise_duration')
    cal_rank = _rank('calorie_expenditure')
    bmi_rank = _rank('bmi')
    water_rank = _rank('water_intake')
    if sleep_rank is not None and step_rank is not None and exercise_rank is not None:
        geom['wellness_rank_core'] = ((sleep_rank + step_rank + exercise_rank) / 3.0).astype('float64')
    if step_rank is not None and exercise_rank is not None and cal_rank is not None:
        geom['activity_rank_core'] = ((step_rank + exercise_rank + cal_rank) / 3.0).astype('float64')
    if bmi_rank is not None and water_rank is not None:
        geom['body_hydration_rank'] = ((1.0 - np.abs(bmi_rank - 0.5) * 2.0) + water_rank).astype('float64')

    geom = geom.replace([np.inf, -np.inf], np.nan)
    train_geom = geom.iloc[:n_train].reset_index(drop=True)
    test_geom = geom.iloc[n_train:].reset_index(drop=True)
    return train_geom, test_geom



X_domain = build_domain_features(X_raw)
X_test_domain = build_domain_features(X_test_raw)
X_fe = add_targeted_interaction_features(X_domain)
X_test_fe = add_targeted_interaction_features(X_test_domain)

X_geom, X_test_geom = build_geometry_forge(X_raw, X_test_raw)
X_geometry = pd.concat([X_domain.reset_index(drop=True), X_geom], axis=1)
X_test_geometry = pd.concat([X_test_domain.reset_index(drop=True), X_test_geom], axis=1)
GEOMETRY_EXTRA_COLS = [c for c in X_geom.columns]
GEOMETRY_NUMERIC_COLS = X_geometry.select_dtypes(include=[np.number, 'bool']).columns.tolist()

DOMAIN_NUMERIC_COLS = X_domain.select_dtypes(include=[np.number, 'bool']).columns.tolist()
INTERACTION_EXTRA_COLS = [c for c in X_fe.columns if c not in X_domain.columns]
model_numeric_cols = X_fe.select_dtypes(include=[np.number, 'bool']).columns.tolist()
categorical_cols = X_fe.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = [c for c in X_fe.columns if c not in categorical_cols]

print('Domain train shape:', X_domain.shape)
print('Geometry train shape:', X_geometry.shape)
print('New geometry columns:', len(GEOMETRY_EXTRA_COLS))
print('Geometry numeric cols:', len(GEOMETRY_NUMERIC_COLS))
print('Domain numeric cols:', len(DOMAIN_NUMERIC_COLS))
display(X_geom.head())


## 4. Validation Utilities

Because our leaderboard evidence favors class-balanced behavior, champion selection prioritizes **balanced accuracy**, then **macro F1**, then ordinary accuracy.

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
le = LabelEncoder()
y_enc = le.fit_transform(y)
classes = list(le.classes_)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}


def prediction_mix(preds: np.ndarray) -> dict:
    counts = pd.Series(preds).value_counts(normalize=True)
    return {f'pred_{cls}_share': float(counts.get(cls, 0.0)) for cls in classes}


def metric_row(name: str, oof_pred: np.ndarray, test_pred: np.ndarray, notes: str = '') -> dict:
    row = {
        'candidate': name,
        'accuracy': accuracy_score(y, oof_pred),
        'balanced_accuracy': balanced_accuracy_score(y, oof_pred),
        'macro_f1': f1_score(y, oof_pred, average='macro'),
        'weighted_f1': f1_score(y, oof_pred, average='weighted'),
        'notes': notes,
    }
    row.update(prediction_mix(test_pred))
    return row


def normalize_rows(proba: np.ndarray) -> np.ndarray:
    proba = np.clip(proba, EPS, None)
    return proba / proba.sum(axis=1, keepdims=True)


def apply_class_multipliers(proba: np.ndarray, fit_multiplier: float = 1.0, unhealthy_multiplier: float = 1.0) -> np.ndarray:
    adjusted = proba.copy()
    if 'fit' in class_to_idx:
        adjusted[:, class_to_idx['fit']] *= fit_multiplier
    if 'unhealthy' in class_to_idx:
        adjusted[:, class_to_idx['unhealthy']] *= unhealthy_multiplier
    return normalize_rows(adjusted)


def proba_to_labels(proba: np.ndarray) -> np.ndarray:
    return le.inverse_transform(np.argmax(proba, axis=1))


def display_diagnostics(name: str, oof_pred: np.ndarray):
    print(f'===== {name} classification report =====')
    print(classification_report(y, oof_pred, digits=4))
    cm = pd.DataFrame(confusion_matrix(y, oof_pred, labels=classes), index=classes, columns=classes)
    print(f'===== {name} confusion matrix =====')
    display(cm)

## 5. Candidate 1: Balanced HGB Anchor

This is the continuity model from our earlier public runs. It gives us a **minority-sensitive anchor** so the new ensemble is not judged only against local accuracy.

In [ ]:
def make_hgb_pipeline() -> Pipeline:
    numeric_transformer = Pipeline(
        steps=[('imputer', SimpleImputer(strategy='median', add_indicator=True))]
    )
    categorical_transformer = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols),
        ],
        remainder='drop',
    )
    model = HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_iter=250,
        l2_regularization=0.01,
        random_state=SEED,
    )
    return Pipeline(steps=[('preprocess', preprocessor), ('model', model)])


def evaluate_hgb_balanced():
    n_classes = len(classes)
    oof_proba = np.zeros((len(X_fe), n_classes), dtype='float64')
    test_proba = np.zeros((len(X_test_fe), n_classes), dtype='float64')

    for fold, (trn_idx, val_idx) in enumerate(cv.split(X_fe, y), 1):
        X_trn, X_val = X_fe.iloc[trn_idx], X_fe.iloc[val_idx]
        y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]
        sample_weight = compute_sample_weight(class_weight='balanced', y=y_trn)

        pipe = make_hgb_pipeline()
        pipe.fit(X_trn, y_trn, model__sample_weight=sample_weight)
        val_proba = pipe.predict_proba(X_val)
        test_proba_fold = pipe.predict_proba(X_test_fe)
        val_pred = proba_to_labels(val_proba)

        oof_proba[val_idx] = val_proba
        test_proba += test_proba_fold / N_SPLITS
        print(f'Fold {fold}: balanced accuracy = {balanced_accuracy_score(y_val, val_pred):.5f}')

    oof_proba = normalize_rows(oof_proba)
    test_proba = normalize_rows(test_proba)
    return proba_to_labels(oof_proba), proba_to_labels(test_proba), oof_proba, test_proba


hgb_oof, hgb_test_pred, hgb_oof_proba, hgb_test_proba = evaluate_hgb_balanced()
display_diagnostics('hgb_balanced_domain', hgb_oof)

## 6. Candidate 2: Balanced LGBM/XGB Domain Ensemble

The improvement idea is simple: use two strong gradient-boosted tree libraries with different inductive biases, then blend their probabilities.

Key differences from the public reference notebook:

- our feature names and composites follow our EDA narrative;
- the blend weight is selected from **OOF validation**, not hard-coded;
- we keep full diagnostics and write comparison artifacts for leaderboard discipline.

In [ ]:
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except Exception as exc:
    lgb = None
    LGBM_AVAILABLE = False
    print('LightGBM unavailable; continuing with XGBoost-only candidate.')
    print(type(exc).__name__, exc)

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except Exception as exc:
    xgb = None
    XGB_AVAILABLE = False
    print('XGBoost unavailable.')
    print(type(exc).__name__, exc)

if not XGB_AVAILABLE:
    raise ImportError('This notebook requires xgboost for the boosted-tree candidate.')

# v8 champion uses the original domain numeric set (no new interaction extras).
X_gbdt = X_domain[DOMAIN_NUMERIC_COLS].astype('float64')
X_test_gbdt = X_test_domain[DOMAIN_NUMERIC_COLS].astype('float64')


V8_LGBM_PARAMS = {
    'n_estimators': 260,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
}
V8_XGB_PARAMS = {
    'n_estimators': 260,
    'learning_rate': 0.05,
    'max_depth': 5,
    'min_child_weight': 2,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'reg_lambda': 1.0,
}


def make_lgbm_model(params=None, random_state=None):
    cfg = {**V8_LGBM_PARAMS, **(params or {})}
    return lgb.LGBMClassifier(
        objective='multiclass',
        class_weight='balanced',
        random_state=SEED if random_state is None else random_state,
        n_jobs=-1,
        verbose=-1,
        **cfg,
    )


def make_xgb_model(params=None, random_state=None):
    cfg = {**V8_XGB_PARAMS, **(params or {})}
    common = dict(
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=SEED if random_state is None else random_state,
        n_jobs=-1,
        **cfg,
    )
    if GPU_AVAILABLE:
        try:
            return xgb.XGBClassifier(tree_method='hist', device='cuda', **common)
        except Exception as exc:
            print('XGB CUDA init failed; falling back to CPU.', type(exc).__name__, exc)
    return xgb.XGBClassifier(tree_method='hist', **common)


def evaluate_lgbm_xgb_ensemble(
    X_train_frame: pd.DataFrame,
    X_test_frame: pd.DataFrame,
    label: str = 'lgbm_xgb',
    lgbm_params=None,
    xgb_params=None,
    random_state=None,
    n_splits=None,
):
    rs = SEED if random_state is None else int(random_state)
    split_count = N_SPLITS if n_splits is None else int(n_splits)
    fold_cv = StratifiedKFold(n_splits=split_count, shuffle=True, random_state=rs)
    n_classes = len(classes)
    oof_lgb = np.zeros((len(X_train_frame), n_classes), dtype='float64') if LGBM_AVAILABLE else None
    oof_xgb = np.zeros((len(X_train_frame), n_classes), dtype='float64')
    test_lgb = np.zeros((len(X_test_frame), n_classes), dtype='float64') if LGBM_AVAILABLE else None
    test_xgb = np.zeros((len(X_test_frame), n_classes), dtype='float64')

    for fold, (trn_idx, val_idx) in enumerate(fold_cv.split(X_train_frame, y), 1):
        X_trn, X_val = X_train_frame.iloc[trn_idx], X_train_frame.iloc[val_idx]
        y_trn_enc, y_val_enc = y_enc[trn_idx], y_enc[val_idx]
        y_val = y.iloc[val_idx]

        fold_parts = []
        if LGBM_AVAILABLE:
            lgb_model = make_lgbm_model(lgbm_params, random_state=rs)
            lgb_model.fit(X_trn, y_trn_enc)
            lgb_val = lgb_model.predict_proba(X_val)
            lgb_test = lgb_model.predict_proba(X_test_frame)
            oof_lgb[val_idx] = lgb_val
            test_lgb += lgb_test / split_count
            lgb_pred = le.inverse_transform(np.argmax(lgb_val, axis=1))
            fold_parts.append(f'LGBM bal_acc={balanced_accuracy_score(y_val, lgb_pred):.5f}')

        xgb_model = make_xgb_model(xgb_params, random_state=rs)
        xgb_weight = compute_sample_weight(class_weight='balanced', y=y_trn_enc)
        xgb_model.fit(X_trn, y_trn_enc, sample_weight=xgb_weight, verbose=False)
        xgb_val = xgb_model.predict_proba(X_val)
        xgb_test_fold = xgb_model.predict_proba(X_test_frame)

        oof_xgb[val_idx] = xgb_val
        test_xgb += xgb_test_fold / split_count
        xgb_pred = le.inverse_transform(np.argmax(xgb_val, axis=1))
        fold_parts.append(f'XGB bal_acc={balanced_accuracy_score(y_val, xgb_pred):.5f}')
        print(f'{label} seed={rs} fold {fold}: ' + ', '.join(fold_parts))

    blend_rows = []
    weight_grid = [0.0, 0.15, 0.30, 0.50, 0.70, 0.85, 1.0] if LGBM_AVAILABLE else [0.0]
    for lgb_weight in weight_grid:
        blend = lgb_weight * oof_lgb + (1.0 - lgb_weight) * oof_xgb if LGBM_AVAILABLE else oof_xgb
        pred = le.inverse_transform(np.argmax(blend, axis=1))
        blend_rows.append(
            {
                'lgb_weight': lgb_weight,
                'xgb_weight': 1.0 - lgb_weight,
                'accuracy': accuracy_score(y, pred),
                'balanced_accuracy': balanced_accuracy_score(y, pred),
                'macro_f1': f1_score(y, pred, average='macro'),
                'weighted_f1': f1_score(y, pred, average='weighted'),
            }
        )

    blend_results = pd.DataFrame(blend_rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)
    best_weight = float(blend_results.loc[0, 'lgb_weight'])

    if LGBM_AVAILABLE:
        oof_blend = best_weight * oof_lgb + (1.0 - best_weight) * oof_xgb
        test_blend = best_weight * test_lgb + (1.0 - best_weight) * test_xgb
    else:
        oof_blend = oof_xgb
        test_blend = test_xgb

    oof_pred = le.inverse_transform(np.argmax(oof_blend, axis=1))
    test_pred = le.inverse_transform(np.argmax(test_blend, axis=1))

    return (
        oof_pred,
        test_pred,
        normalize_rows(oof_blend),
        normalize_rows(test_blend),
        blend_results,
        best_weight,
        None if oof_lgb is None else normalize_rows(oof_lgb),
        None if test_lgb is None else normalize_rows(test_lgb),
        normalize_rows(oof_xgb),
        normalize_rows(test_xgb),
    )


(
    ensemble_oof,
    ensemble_test_pred,
    ensemble_oof_proba,
    ensemble_test_proba,
    blend_results,
    best_lgb_weight,
    domain_oof_lgb,
    domain_test_lgb,
    domain_oof_xgb,
    domain_test_xgb,
) = evaluate_lgbm_xgb_ensemble(X_gbdt, X_test_gbdt, label='lgbm_xgb_domain')

print('Best OOF blend weight:')
print({'lgb_weight': best_lgb_weight, 'xgb_weight': 1.0 - best_lgb_weight})
display(blend_results)
display_diagnostics('lgbm_xgb_domain_ensemble', ensemble_oof)


## 7. HGB + LGBM/XGB Probability Blend

The v8 champion blends **LightGBM and XGBoost** only. The balanced HGB anchor uses a different preprocessing path and may capture complementary errors. We sweep a convex mix of HGB probabilities with the selected LGBM/XGB blend and keep the weight that maximizes OOF balanced accuracy.

In [ ]:
if RUN_EXPLORATORY_ABLATIONS:
    def sweep_hgb_lgbm_xgb_blend():
        rows = []
        hgb_weight_grid = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.85, 1.0]

        for hgb_weight in hgb_weight_grid:
            ensemble_weight = 1.0 - hgb_weight
            oof_blend = hgb_weight * hgb_oof_proba + ensemble_weight * ensemble_oof_proba
            test_blend = hgb_weight * hgb_test_proba + ensemble_weight * ensemble_test_proba
            oof_blend = normalize_rows(oof_blend)
            test_blend = normalize_rows(test_blend)
            oof_pred = proba_to_labels(oof_blend)
            test_pred = proba_to_labels(test_blend)
            rows.append(
                {
                    'hgb_weight': hgb_weight,
                    'ensemble_weight': ensemble_weight,
                    'accuracy': accuracy_score(y, oof_pred),
                    'balanced_accuracy': balanced_accuracy_score(y, oof_pred),
                    'macro_f1': f1_score(y, oof_pred, average='macro'),
                    'weighted_f1': f1_score(y, oof_pred, average='weighted'),
                }
            )

        hgb_blend_results = pd.DataFrame(rows).sort_values(
            ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
        ).reset_index(drop=True)

        best_hgb_weight = float(hgb_blend_results.loc[0, 'hgb_weight'])
        best_ensemble_weight = 1.0 - best_hgb_weight
        best_oof_proba = normalize_rows(best_hgb_weight * hgb_oof_proba + best_ensemble_weight * ensemble_oof_proba)
        best_test_proba = normalize_rows(best_hgb_weight * hgb_test_proba + best_ensemble_weight * ensemble_test_proba)
        best_oof_pred = proba_to_labels(best_oof_proba)
        best_test_pred = proba_to_labels(best_test_proba)
        return hgb_blend_results, best_oof_pred, best_test_pred, best_hgb_weight


    hgb_blend_results, hgb_ensemble_oof, hgb_ensemble_test_pred, best_hgb_weight = sweep_hgb_lgbm_xgb_blend()

    print('Best HGB + LGBM/XGB blend weight:')
    print({'hgb_weight': best_hgb_weight, 'ensemble_weight': 1.0 - best_hgb_weight})
    display(hgb_blend_results)
    hgb_blend_results.to_csv(WORK_DIR / 'hgb_ensemble_blend_results.csv', index=False)
    display_diagnostics('hgb_lgbm_xgb_domain_blend', hgb_ensemble_oof)
else:
    print('Skipping HGB blend ablation (previously failed champion gate).')
    hgb_blend_results = pd.DataFrame(
        columns=['hgb_weight', 'ensemble_weight', 'accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1']
    )
    hgb_ensemble_oof = ensemble_oof
    hgb_ensemble_test_pred = ensemble_test_pred
    best_hgb_weight = 0.0
    pd.DataFrame().to_csv(WORK_DIR / 'hgb_ensemble_blend_results.csv', index=False)


## 7b. Targeted Interaction Features

v13 showed that blending HGB probabilities into the v8 ensemble barely moved balanced accuracy
(`+0.000003`). The next hypothesis is **feature composition**, not blend weight.

We keep the domain feature pack as the v8 continuity path, then add a small set of row-safe
cross features (`sleep_activity_score`, `stress_sleep_risk`, mismatch / deficit scores, and
key missing flags) and retrain the same balanced LGBM/XGB blend.

Promotion still uses the champion gate versus `lgbm_xgb_domain_ensemble`.


In [ ]:
if RUN_EXPLORATORY_ABLATIONS:
    X_interact = X_fe[model_numeric_cols].astype('float64')
    X_test_interact = X_test_fe[model_numeric_cols].astype('float64')

    (
        interaction_oof,
        interaction_test_pred,
        interaction_oof_proba,
        interaction_test_proba,
        interaction_blend_results,
        best_interaction_lgb_weight,
    ) = evaluate_lgbm_xgb_ensemble(X_interact, X_test_interact, label='lgbm_xgb_interaction')

    print('Best interaction OOF blend weight:')
    print({
        'lgb_weight': best_interaction_lgb_weight,
        'xgb_weight': 1.0 - best_interaction_lgb_weight,
        'n_features': X_interact.shape[1],
        'n_new_features': len(INTERACTION_EXTRA_COLS),
    })
    display(interaction_blend_results)
    interaction_blend_results.to_csv(WORK_DIR / 'interaction_blend_results.csv', index=False)
    display_diagnostics('lgbm_xgb_interaction_ensemble', interaction_oof)
else:
    print('Skipping interaction FE ablation (previously failed champion gate).')
    interaction_oof = ensemble_oof
    interaction_test_pred = ensemble_test_pred
    interaction_oof_proba = ensemble_oof_proba
    interaction_test_proba = ensemble_test_proba
    interaction_blend_results = blend_results.copy()
    best_interaction_lgb_weight = best_lgb_weight
    INTERACTION_EXTRA_COLS = INTERACTION_EXTRA_COLS  # keep defined from FE
    pd.DataFrame().to_csv(WORK_DIR / 'interaction_blend_results.csv', index=False)


## 7c. Focused LGBM/XGB Hyperparameter Search (skipped)

v15 already tested deeper/longer LGBM/XGB configs on the domain feature set. The best
tuned blend gained only `+0.000041` balanced accuracy — below the champion gate — so
HP search stays disabled by default (`RUN_HP_SEARCH = False`).


In [ ]:
HP_LGBM_CONFIGS = [
    {'name': 'deeper', 'n_estimators': 420, 'learning_rate': 0.04, 'num_leaves': 63, 'subsample': 0.9, 'colsample_bytree': 0.85},
    {'name': 'longer', 'n_estimators': 560, 'learning_rate': 0.03, 'num_leaves': 31, 'subsample': 0.85, 'colsample_bytree': 0.9},
]

HP_XGB_CONFIGS = [
    {'name': 'deeper', 'n_estimators': 420, 'learning_rate': 0.04, 'max_depth': 6, 'min_child_weight': 2, 'subsample': 0.9, 'colsample_bytree': 0.85, 'reg_lambda': 1.2},
    {'name': 'longer', 'n_estimators': 560, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 3, 'subsample': 0.85, 'colsample_bytree': 0.9, 'reg_lambda': 1.5},
]


def _drop_name(cfg: dict) -> dict:
    return {k: v for k, v in cfg.items() if k != 'name'}


def evaluate_single_model_oof(model_factory, X_train_frame, X_test_frame, label: str):
    n_classes = len(classes)
    oof = np.zeros((len(X_train_frame), n_classes), dtype='float64')
    test = np.zeros((len(X_test_frame), n_classes), dtype='float64')
    for fold, (trn_idx, val_idx) in enumerate(cv.split(X_train_frame, y), 1):
        X_trn, X_val = X_train_frame.iloc[trn_idx], X_train_frame.iloc[val_idx]
        y_trn_enc = y_enc[trn_idx]
        model = model_factory()
        if isinstance(model, xgb.XGBClassifier):
            sw = compute_sample_weight(class_weight='balanced', y=y_trn_enc)
            model.fit(X_trn, y_trn_enc, sample_weight=sw, verbose=False)
        else:
            model.fit(X_trn, y_trn_enc)
        oof[val_idx] = model.predict_proba(X_val)
        test += model.predict_proba(X_test_frame) / N_SPLITS
        pred = le.inverse_transform(np.argmax(oof[val_idx], axis=1))
        print(f'{label} fold {fold}: bal_acc={balanced_accuracy_score(y.iloc[val_idx], pred):.5f}')
    return normalize_rows(oof), normalize_rows(test)


if RUN_HP_SEARCH:
    lgbm_oof_bank = {}
    lgbm_test_bank = {}
    xgb_oof_bank = {}
    xgb_test_bank = {}
    hp_model_rows = []

    # Reuse the already-trained v8 domain components when available.
    if LGBM_AVAILABLE and domain_oof_lgb is not None:
        lgbm_oof_bank['v8'] = domain_oof_lgb
        lgbm_test_bank['v8'] = domain_test_lgb
        pred = le.inverse_transform(np.argmax(domain_oof_lgb, axis=1))
        hp_model_rows.append(
            {
                'family': 'lgbm',
                'config': 'v8',
                'balanced_accuracy': balanced_accuracy_score(y, pred),
                'macro_f1': f1_score(y, pred, average='macro'),
                'accuracy': accuracy_score(y, pred),
                **V8_LGBM_PARAMS,
            }
        )
    xgb_oof_bank['v8'] = domain_oof_xgb
    xgb_test_bank['v8'] = domain_test_xgb
    pred = le.inverse_transform(np.argmax(domain_oof_xgb, axis=1))
    hp_model_rows.append(
        {
            'family': 'xgb',
            'config': 'v8',
            'balanced_accuracy': balanced_accuracy_score(y, pred),
            'macro_f1': f1_score(y, pred, average='macro'),
            'accuracy': accuracy_score(y, pred),
            **V8_XGB_PARAMS,
        }
    )

    for cfg in HP_LGBM_CONFIGS:
        if not LGBM_AVAILABLE:
            break
        name = cfg['name']
        print(f'=== LGBM config: {name} ===')
        params = _drop_name(cfg)
        oof, test = evaluate_single_model_oof(
            lambda p=params: make_lgbm_model(p),
            X_gbdt,
            X_test_gbdt,
            label=f'lgbm_{name}',
        )
        lgbm_oof_bank[name] = oof
        lgbm_test_bank[name] = test
        pred = le.inverse_transform(np.argmax(oof, axis=1))
        hp_model_rows.append(
            {
                'family': 'lgbm',
                'config': name,
                'balanced_accuracy': balanced_accuracy_score(y, pred),
                'macro_f1': f1_score(y, pred, average='macro'),
                'accuracy': accuracy_score(y, pred),
                **params,
            }
        )

    for cfg in HP_XGB_CONFIGS:
        name = cfg['name']
        device_name = 'cuda' if GPU_AVAILABLE else 'cpu'
        print(f'=== XGB config: {name} (device={device_name}) ===')
        params = _drop_name(cfg)
        oof, test = evaluate_single_model_oof(
            lambda p=params: make_xgb_model(p),
            X_gbdt,
            X_test_gbdt,
            label=f'xgb_{name}',
        )
        xgb_oof_bank[name] = oof
        xgb_test_bank[name] = test
        pred = le.inverse_transform(np.argmax(oof, axis=1))
        hp_model_rows.append(
            {
                'family': 'xgb',
                'config': name,
                'balanced_accuracy': balanced_accuracy_score(y, pred),
                'macro_f1': f1_score(y, pred, average='macro'),
                'accuracy': accuracy_score(y, pred),
                **params,
            }
        )

    hp_model_results = pd.DataFrame(hp_model_rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)
    display(hp_model_results)
    hp_model_results.to_csv(WORK_DIR / 'hp_model_results.csv', index=False)

    blend_rows = []
    lgb_names = list(lgbm_oof_bank) if lgbm_oof_bank else [None]
    for lgb_name in lgb_names:
        for xgb_name, xgb_oof in xgb_oof_bank.items():
            weight_grid = [0.0, 0.15, 0.30, 0.50, 0.70, 0.85, 1.0] if lgb_name is not None else [0.0]
            for lgb_weight in weight_grid:
                if lgb_name is None:
                    blend = xgb_oof
                    test_blend = xgb_test_bank[xgb_name]
                else:
                    blend = lgb_weight * lgbm_oof_bank[lgb_name] + (1.0 - lgb_weight) * xgb_oof
                    test_blend = (
                        lgb_weight * lgbm_test_bank[lgb_name]
                        + (1.0 - lgb_weight) * xgb_test_bank[xgb_name]
                    )
                pred = le.inverse_transform(np.argmax(blend, axis=1))
                blend_rows.append(
                    {
                        'lgbm_config': lgb_name,
                        'xgb_config': xgb_name,
                        'lgb_weight': lgb_weight,
                        'xgb_weight': 1.0 - lgb_weight,
                        'accuracy': accuracy_score(y, pred),
                        'balanced_accuracy': balanced_accuracy_score(y, pred),
                        'macro_f1': f1_score(y, pred, average='macro'),
                        'weighted_f1': f1_score(y, pred, average='weighted'),
                        'oof_blend': blend,
                        'test_blend': test_blend,
                    }
                )

    tuned_blend_results = pd.DataFrame(blend_rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)
    best = tuned_blend_results.loc[0]
    tuned_oof_proba = normalize_rows(best['oof_blend'])
    tuned_test_proba = normalize_rows(best['test_blend'])
    tuned_oof = le.inverse_transform(np.argmax(tuned_oof_proba, axis=1))
    tuned_test_pred = le.inverse_transform(np.argmax(tuned_test_proba, axis=1))
    best_tuned_lgbm = best['lgbm_config']
    best_tuned_xgb = best['xgb_config']
    best_tuned_lgb_weight = float(best['lgb_weight'])

    tuned_blend_save = tuned_blend_results.drop(columns=['oof_blend', 'test_blend'])
    display(tuned_blend_save.head(15))
    tuned_blend_save.to_csv(WORK_DIR / 'hp_blend_results.csv', index=False)
    print(
        'Best tuned blend:',
        {
            'lgbm_config': best_tuned_lgbm,
            'xgb_config': best_tuned_xgb,
            'lgb_weight': best_tuned_lgb_weight,
            'balanced_accuracy': float(best['balanced_accuracy']),
            'macro_f1': float(best['macro_f1']),
        },
    )
    display_diagnostics('lgbm_xgb_tuned_domain_ensemble', tuned_oof)
else:
    print('HP search disabled.')
    tuned_oof = ensemble_oof
    tuned_test_pred = ensemble_test_pred
    tuned_oof_proba = ensemble_oof_proba
    tuned_test_proba = ensemble_test_proba
    best_tuned_lgbm = 'v8'
    best_tuned_xgb = 'v8'
    best_tuned_lgb_weight = best_lgb_weight
    tuned_blend_save = pd.DataFrame()
    hp_model_results = pd.DataFrame()
    pd.DataFrame().to_csv(WORK_DIR / 'hp_model_results.csv', index=False)
    pd.DataFrame().to_csv(WORK_DIR / 'hp_blend_results.csv', index=False)


## 7d. Multi-Seed LGBM/XGB Averaging

Small local grids around the v8 recipe (calibration, HGB blend, interactions, HP) all
failed the `0.0002` gate. This experiment keeps the **same domain features, class balance,
and model params**, and averages OOF/test probabilities across several seeds.

For each seed we:

1. rebuild the stratified 3-fold split with that seed;
2. retrain balanced LGBM and XGBoost with that seed;
3. accumulate component probabilities;
4. average across seeds, then re-sweep the LGBM/XGB blend weight on the averaged OOF.

The seed-42 run reuses the already-trained v8 component probabilities to avoid duplicate work.
Candidate name: `lgbm_xgb_multiseed_domain_ensemble`.


In [ ]:
def evaluate_multiseed_lgbm_xgb_ensemble(
    X_train_frame: pd.DataFrame,
    X_test_frame: pd.DataFrame,
    seeds=None,
):
    """Average LGBM/XGB OOF and test probabilities across seeds, then blend."""
    seeds = list(MULTI_SEEDS if seeds is None else seeds)
    n_classes = len(classes)
    n_seeds = len(seeds)

    oof_lgb_sum = np.zeros((len(X_train_frame), n_classes), dtype='float64') if LGBM_AVAILABLE else None
    oof_xgb_sum = np.zeros((len(X_train_frame), n_classes), dtype='float64')
    test_lgb_sum = np.zeros((len(X_test_frame), n_classes), dtype='float64') if LGBM_AVAILABLE else None
    test_xgb_sum = np.zeros((len(X_test_frame), n_classes), dtype='float64')
    seed_rows = []

    for seed in seeds:
        if seed == SEED and domain_oof_xgb is not None:
            print(f'Reusing v8 seed={seed} component probabilities.')
            oof_lgb = domain_oof_lgb
            test_lgb = domain_test_lgb
            oof_xgb = domain_oof_xgb
            test_xgb = domain_test_xgb
            if LGBM_AVAILABLE:
                blend = best_lgb_weight * oof_lgb + (1.0 - best_lgb_weight) * oof_xgb
            else:
                blend = oof_xgb
            seed_pred = le.inverse_transform(np.argmax(blend, axis=1))
            seed_bal = balanced_accuracy_score(y, seed_pred)
            seed_macro = f1_score(y, seed_pred, average='macro')
        else:
            (
                seed_pred,
                _,
                _,
                _,
                seed_blend_results,
                seed_best_weight,
                oof_lgb,
                test_lgb,
                oof_xgb,
                test_xgb,
            ) = evaluate_lgbm_xgb_ensemble(
                X_train_frame,
                X_test_frame,
                label='lgbm_xgb_multiseed',
                random_state=seed,
            )
            seed_bal = float(seed_blend_results.loc[0, 'balanced_accuracy'])
            seed_macro = float(seed_blend_results.loc[0, 'macro_f1'])
            seed_best_weight = float(seed_best_weight)

        if LGBM_AVAILABLE:
            oof_lgb_sum += oof_lgb
            test_lgb_sum += test_lgb
        oof_xgb_sum += oof_xgb
        test_xgb_sum += test_xgb

        seed_rows.append(
            {
                'seed': seed,
                'balanced_accuracy': seed_bal,
                'macro_f1': seed_macro,
                'reused_v8_components': bool(seed == SEED),
            }
        )
        print(
            f'Multi-seed summary seed={seed}: '
            f'bal_acc={seed_bal:.5f}, macro_f1={seed_macro:.5f}'
        )

    if LGBM_AVAILABLE:
        oof_lgb_avg = normalize_rows(oof_lgb_sum / n_seeds)
        test_lgb_avg = normalize_rows(test_lgb_sum / n_seeds)
    else:
        oof_lgb_avg = None
        test_lgb_avg = None
    oof_xgb_avg = normalize_rows(oof_xgb_sum / n_seeds)
    test_xgb_avg = normalize_rows(test_xgb_sum / n_seeds)

    blend_rows = []
    weight_grid = [0.0, 0.15, 0.30, 0.50, 0.70, 0.85, 1.0] if LGBM_AVAILABLE else [0.0]
    for lgb_weight in weight_grid:
        blend = (
            lgb_weight * oof_lgb_avg + (1.0 - lgb_weight) * oof_xgb_avg
            if LGBM_AVAILABLE
            else oof_xgb_avg
        )
        pred = le.inverse_transform(np.argmax(blend, axis=1))
        blend_rows.append(
            {
                'lgb_weight': lgb_weight,
                'xgb_weight': 1.0 - lgb_weight,
                'accuracy': accuracy_score(y, pred),
                'balanced_accuracy': balanced_accuracy_score(y, pred),
                'macro_f1': f1_score(y, pred, average='macro'),
                'weighted_f1': f1_score(y, pred, average='weighted'),
            }
        )

    blend_results = pd.DataFrame(blend_rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)
    best_weight = float(blend_results.loc[0, 'lgb_weight'])

    if LGBM_AVAILABLE:
        oof_blend = best_weight * oof_lgb_avg + (1.0 - best_weight) * oof_xgb_avg
        test_blend = best_weight * test_lgb_avg + (1.0 - best_weight) * test_xgb_avg
    else:
        oof_blend = oof_xgb_avg
        test_blend = test_xgb_avg

    oof_pred = le.inverse_transform(np.argmax(oof_blend, axis=1))
    test_pred = le.inverse_transform(np.argmax(test_blend, axis=1))
    seed_summary = pd.DataFrame(seed_rows)

    return (
        oof_pred,
        test_pred,
        normalize_rows(oof_blend),
        normalize_rows(test_blend),
        blend_results,
        best_weight,
        seed_summary,
    )


if RUN_MULTI_SEED:
    (
        multiseed_oof,
        multiseed_test_pred,
        multiseed_oof_proba,
        multiseed_test_proba,
        multiseed_blend_results,
        best_multiseed_lgb_weight,
        multiseed_seed_summary,
    ) = evaluate_multiseed_lgbm_xgb_ensemble(X_gbdt, X_test_gbdt, seeds=MULTI_SEEDS)

    print('Per-seed OOF summary:')
    display(multiseed_seed_summary)
    print('Multi-seed averaged blend sweep:')
    display(multiseed_blend_results)
    print(
        'Best multi-seed blend:',
        {
            'seeds': MULTI_SEEDS,
            'lgb_weight': best_multiseed_lgb_weight,
            'balanced_accuracy': float(multiseed_blend_results.loc[0, 'balanced_accuracy']),
            'macro_f1': float(multiseed_blend_results.loc[0, 'macro_f1']),
        },
    )
    display_diagnostics('lgbm_xgb_multiseed_domain_ensemble', multiseed_oof)
    multiseed_seed_summary.to_csv(WORK_DIR / 'multiseed_seed_summary.csv', index=False)
    multiseed_blend_results.to_csv(WORK_DIR / 'multiseed_blend_results.csv', index=False)
else:
    print('Multi-seed averaging disabled.')
    multiseed_oof = ensemble_oof
    multiseed_test_pred = ensemble_test_pred
    multiseed_oof_proba = ensemble_oof_proba
    multiseed_test_proba = ensemble_test_proba
    best_multiseed_lgb_weight = best_lgb_weight
    multiseed_blend_results = pd.DataFrame()
    multiseed_seed_summary = pd.DataFrame()
    pd.DataFrame().to_csv(WORK_DIR / 'multiseed_seed_summary.csv', index=False)
    pd.DataFrame().to_csv(WORK_DIR / 'multiseed_blend_results.csv', index=False)


## 7e. Five-Fold Stability And Cross-Fitted Thresholds

v16 multi-seed averaging improved OOF balanced accuracy by only `+0.000029`, so this
version tests two larger validation changes:

1. **5-fold v8 LGBM/XGB**: train each fold on 80% rather than 66.7% of the rows,
   then select the LGBM/XGB weight from 5-fold OOF predictions.
2. **Cross-fitted rare-class thresholds**: on each calibration fold, select `fit` and
   `unhealthy` multipliers using only the other folds, then apply them to the untouched
   fold. This produces leakage-safe OOF predictions rather than scoring thresholds on
   the same rows used to choose them.

Both candidates must still clear the unchanged champion gate.


In [ ]:
if RUN_FIVE_FOLD:
    (
        fivefold_oof,
        fivefold_test_pred,
        fivefold_oof_proba,
        fivefold_test_proba,
        fivefold_blend_results,
        best_fivefold_lgb_weight,
        _,
        _,
        _,
        _,
    ) = evaluate_lgbm_xgb_ensemble(
        X_gbdt,
        X_test_gbdt,
        label='lgbm_xgb_fivefold',
        random_state=SEED,
        n_splits=5,
    )
    display(fivefold_blend_results)
    display_diagnostics('lgbm_xgb_fivefold_domain_ensemble', fivefold_oof)
    fivefold_blend_results.to_csv(WORK_DIR / 'fivefold_blend_results.csv', index=False)
else:
    print('Five-fold experiment disabled.')
    fivefold_oof = ensemble_oof
    fivefold_test_pred = ensemble_test_pred
    fivefold_oof_proba = ensemble_oof_proba
    fivefold_test_proba = ensemble_test_proba
    best_fivefold_lgb_weight = best_lgb_weight
    fivefold_blend_results = pd.DataFrame()
    pd.DataFrame().to_csv(WORK_DIR / 'fivefold_blend_results.csv', index=False)


def select_class_multipliers(y_true, proba):
    """Select rare-class multipliers on training rows only."""
    rows = []
    for fit_multiplier in [0.92, 0.96, 1.00, 1.04, 1.08, 1.12]:
        for unhealthy_multiplier in [0.92, 0.96, 1.00, 1.04, 1.08, 1.12]:
            adjusted = apply_class_multipliers(proba, fit_multiplier, unhealthy_multiplier)
            pred = proba_to_labels(adjusted)
            rows.append(
                {
                    'fit_multiplier': fit_multiplier,
                    'unhealthy_multiplier': unhealthy_multiplier,
                    'balanced_accuracy': balanced_accuracy_score(y_true, pred),
                    'macro_f1': f1_score(y_true, pred, average='macro'),
                    'accuracy': accuracy_score(y_true, pred),
                }
            )
    return pd.DataFrame(rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)


def evaluate_crossfit_thresholds(base_oof_proba, base_test_proba):
    """Tune multipliers off-fold and apply them only to untouched OOF rows."""
    calibration_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 101)
    adjusted_oof = np.zeros_like(base_oof_proba)
    fold_rows = []

    for fold, (tune_idx, holdout_idx) in enumerate(calibration_cv.split(base_oof_proba, y), 1):
        sweep = select_class_multipliers(y.iloc[tune_idx], base_oof_proba[tune_idx])
        best = sweep.iloc[0]
        fit_multiplier = float(best['fit_multiplier'])
        unhealthy_multiplier = float(best['unhealthy_multiplier'])
        adjusted_oof[holdout_idx] = apply_class_multipliers(
            base_oof_proba[holdout_idx], fit_multiplier, unhealthy_multiplier
        )
        holdout_pred = proba_to_labels(adjusted_oof[holdout_idx])
        fold_rows.append(
            {
                'fold': fold,
                'fit_multiplier': fit_multiplier,
                'unhealthy_multiplier': unhealthy_multiplier,
                'tune_balanced_accuracy': float(best['balanced_accuracy']),
                'holdout_balanced_accuracy': balanced_accuracy_score(y.iloc[holdout_idx], holdout_pred),
                'holdout_macro_f1': f1_score(y.iloc[holdout_idx], holdout_pred, average='macro'),
            }
        )

    fold_results = pd.DataFrame(fold_rows)
    test_fit_multiplier = float(fold_results['fit_multiplier'].median())
    test_unhealthy_multiplier = float(fold_results['unhealthy_multiplier'].median())
    adjusted_test = apply_class_multipliers(
        base_test_proba, test_fit_multiplier, test_unhealthy_multiplier
    )
    return (
        proba_to_labels(adjusted_oof),
        proba_to_labels(adjusted_test),
        normalize_rows(adjusted_oof),
        normalize_rows(adjusted_test),
        fold_results,
        test_fit_multiplier,
        test_unhealthy_multiplier,
    )


if RUN_CROSSFIT_THRESHOLDS:
    (
        crossfit_threshold_oof,
        crossfit_threshold_test_pred,
        crossfit_threshold_oof_proba,
        crossfit_threshold_test_proba,
        crossfit_threshold_results,
        crossfit_test_fit_multiplier,
        crossfit_test_unhealthy_multiplier,
    ) = evaluate_crossfit_thresholds(ensemble_oof_proba, ensemble_test_proba)
    display(crossfit_threshold_results)
    display_diagnostics('crossfit_threshold_lgbm_xgb', crossfit_threshold_oof)
    crossfit_threshold_results.to_csv(WORK_DIR / 'crossfit_threshold_results.csv', index=False)
else:
    print('Cross-fitted threshold experiment disabled.')
    crossfit_threshold_oof = ensemble_oof
    crossfit_threshold_test_pred = ensemble_test_pred
    crossfit_threshold_oof_proba = ensemble_oof_proba
    crossfit_threshold_test_proba = ensemble_test_proba
    crossfit_test_fit_multiplier = 1.0
    crossfit_test_unhealthy_multiplier = 1.0
    crossfit_threshold_results = pd.DataFrame()
    pd.DataFrame().to_csv(WORK_DIR / 'crossfit_threshold_results.csv', index=False)


## 7f. Synthetic-Geometry LGBM/XGB Ensemble

Strong public notebooks on this synthetic tabular task gain lift from **rank geometry**,
quantile categories, and conditional numeric deviations rather than more blending.
This section keeps the v8 class-balanced LGBM/XGB recipe but retrains on
`domain + geometry` numeric features.

Candidate: `lgbm_xgb_geometry_ensemble`.


In [ ]:
if RUN_GEOMETRY_FORGE:
    X_geom_gbdt = X_geometry[GEOMETRY_NUMERIC_COLS].astype('float64')
    X_test_geom_gbdt = X_test_geometry[GEOMETRY_NUMERIC_COLS].astype('float64')
    (
        geometry_oof,
        geometry_test_pred,
        geometry_oof_proba,
        geometry_test_proba,
        geometry_blend_results,
        best_geometry_lgb_weight,
        _,
        _,
        _,
        _,
    ) = evaluate_lgbm_xgb_ensemble(
        X_geom_gbdt,
        X_test_geom_gbdt,
        label='lgbm_xgb_geometry',
    )
    display(geometry_blend_results)
    display_diagnostics('lgbm_xgb_geometry_ensemble', geometry_oof)
    geometry_blend_results.to_csv(WORK_DIR / 'geometry_blend_results.csv', index=False)
else:
    print('Geometry forge disabled.')
    geometry_oof = ensemble_oof
    geometry_test_pred = ensemble_test_pred
    geometry_oof_proba = ensemble_oof_proba
    geometry_test_proba = ensemble_test_proba
    best_geometry_lgb_weight = best_lgb_weight
    geometry_blend_results = pd.DataFrame()
    pd.DataFrame().to_csv(WORK_DIR / 'geometry_blend_results.csv', index=False)


## 8. Calibration And Continuity Checks

The v8 champion remains the LGBM/XGB domain blend. This section keeps a small
class-prior calibration sweep as a continuity check while the active v19 experiment
runs in section 7f.


In [ ]:
def sweep_class_prior_calibration():
    rows = []
    fit_multipliers = [0.88, 0.92, 0.96, 1.00, 1.04, 1.08, 1.12]
    unhealthy_multipliers = [0.88, 0.92, 0.96, 1.00, 1.04, 1.08, 1.12]

    for fit_multiplier in fit_multipliers:
        for unhealthy_multiplier in unhealthy_multipliers:
            adjusted_oof = apply_class_multipliers(ensemble_oof_proba, fit_multiplier, unhealthy_multiplier)
            adjusted_test = apply_class_multipliers(ensemble_test_proba, fit_multiplier, unhealthy_multiplier)
            oof_pred = proba_to_labels(adjusted_oof)
            test_pred = proba_to_labels(adjusted_test)
            row = metric_row(
                'calibrated_lgbm_xgb_domain_ensemble',
                oof_pred,
                test_pred,
                notes=(
                    f'LGBM/XGB probability blend; fit multiplier={fit_multiplier:.2f}; '
                    f'unhealthy multiplier={unhealthy_multiplier:.2f}.'
                ),
            )
            row.update(
                {
                    'fit_multiplier': fit_multiplier,
                    'unhealthy_multiplier': unhealthy_multiplier,
                }
            )
            rows.append(row)

    calibration_results = pd.DataFrame(rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)

    best = calibration_results.loc[0]
    best_oof_proba = apply_class_multipliers(ensemble_oof_proba, best['fit_multiplier'], best['unhealthy_multiplier'])
    best_test_proba = apply_class_multipliers(ensemble_test_proba, best['fit_multiplier'], best['unhealthy_multiplier'])

    best_oof_pred = proba_to_labels(best_oof_proba)
    best_test_pred = proba_to_labels(best_test_proba)
    return calibration_results, best_oof_pred, best_test_pred, best_oof_proba, best_test_proba


def prepare_catboost_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in categorical_cols:
        out[col] = out[col].fillna('__missing__').astype(str)
    return out


try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception as exc:
    CatBoostClassifier = None
    CATBOOST_AVAILABLE = False
    print('CatBoost unavailable; skipping CatBoost signal engine.')
    print(type(exc).__name__, exc)


def make_catboost_model():
    return CatBoostClassifier(
        loss_function='MultiClass',
        eval_metric='Accuracy',
        iterations=160,
        depth=7,
        learning_rate=0.08,
        l2_leaf_reg=7.0,
        random_seed=SEED,
        random_strength=0.35,
        bootstrap_type='Bernoulli',
        subsample=0.85,
        auto_class_weights='Balanced',
        allow_writing_files=False,
        task_type=('GPU' if GPU_AVAILABLE else 'CPU'),
        thread_count=-1,
        verbose=False,
    )


def evaluate_catboost_native():
    if not CATBOOST_AVAILABLE:
        return None, None, None, None

    X_cat = prepare_catboost_frame(X_fe)
    X_test_cat = prepare_catboost_frame(X_test_fe)
    n_classes = len(classes)
    oof_proba = np.zeros((len(X_cat), n_classes), dtype='float64')
    test_proba = np.zeros((len(X_test_cat), n_classes), dtype='float64')

    for fold, (trn_idx, val_idx) in enumerate(cv.split(X_cat, y), 1):
        X_trn, X_val = X_cat.iloc[trn_idx], X_cat.iloc[val_idx]
        y_trn_enc = y_enc[trn_idx]
        y_val = y.iloc[val_idx]

        model = make_catboost_model()
        model.fit(X_trn, y_trn_enc, cat_features=categorical_cols)
        val_proba = model.predict_proba(X_val)
        test_proba_fold = model.predict_proba(X_test_cat)

        oof_proba[val_idx] = val_proba
        test_proba += test_proba_fold / N_SPLITS

        val_pred = proba_to_labels(val_proba)
        print(f'Fold {fold}: CatBoost balanced accuracy = {balanced_accuracy_score(y_val, val_pred):.5f}')

    oof_proba = normalize_rows(oof_proba)
    test_proba = normalize_rows(test_proba)
    return proba_to_labels(oof_proba), proba_to_labels(test_proba), oof_proba, test_proba


def build_signal_predictions(base_proba, arbiter_proba, gate):
    max_flips = int(gate['max_flips'])
    base_idx = np.argmax(base_proba, axis=1)
    arbiter_idx = np.argmax(arbiter_proba, axis=1)
    row = np.arange(len(base_idx))

    base_conf = base_proba[row, base_idx]
    arbiter_conf = arbiter_proba[row, arbiter_idx]
    arbiter_margin = arbiter_conf - arbiter_proba[row, base_idx]

    candidate = (
        (arbiter_idx != base_idx) &
        (base_conf <= gate['max_base_conf']) &
        (arbiter_conf >= gate['min_arbiter_conf']) &
        (arbiter_margin >= gate['min_margin'])
    )

    score = 0.55 * arbiter_conf + 0.30 * np.clip(arbiter_margin, 0, 1) + 0.15 * (1.0 - base_conf)
    candidate_rows = np.flatnonzero(candidate)
    if max_flips > 0 and len(candidate_rows) > max_flips:
        candidate_rows = candidate_rows[np.argsort(-score[candidate_rows], kind='stable')[:max_flips]]

    final_idx = base_idx.copy()
    final_idx[candidate_rows] = arbiter_idx[candidate_rows]
    return le.inverse_transform(final_idx), len(candidate_rows)


def sweep_catboost_signal_engine():
    if not CATBOOST_AVAILABLE:
        empty = pd.DataFrame()
        return empty, ensemble_oof, ensemble_test_pred

    rows = []
    gates = []
    for max_base_conf in [0.65, 0.70]:
        for min_arbiter_conf in [0.62, 0.74]:
            for min_margin in [0.08, 0.16]:
                for max_flips in [0, 25, 100, 500]:
                    gates.append(
                        {
                            'max_base_conf': max_base_conf,
                            'min_arbiter_conf': min_arbiter_conf,
                            'min_margin': min_margin,
                            'max_flips': max_flips,
                        }
                    )

    for gate in gates:
        oof_pred, oof_flips = build_signal_predictions(ensemble_oof_proba, catboost_oof_proba, gate)
        test_pred, test_flips = build_signal_predictions(ensemble_test_proba, catboost_test_proba, gate)
        row = metric_row(
            'catboost_signal_engine',
            oof_pred,
            test_pred,
            notes=(
                f"OOF-gated micro-flips: max_base_conf={gate['max_base_conf']:.2f}; "
                f"min_cat_conf={gate['min_arbiter_conf']:.2f}; min_margin={gate['min_margin']:.2f}; "
                f"max_flips={gate['max_flips']}."
            ),
        )
        row.update(gate)
        row['oof_flips'] = oof_flips
        row['test_flips'] = test_flips
        rows.append(row)

    signal_results = pd.DataFrame(rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)

    best_gate = signal_results.loc[0, ['max_base_conf', 'min_arbiter_conf', 'min_margin', 'max_flips']].to_dict()
    best_oof_pred, _ = build_signal_predictions(ensemble_oof_proba, catboost_oof_proba, best_gate)
    best_test_pred, _ = build_signal_predictions(ensemble_test_proba, catboost_test_proba, best_gate)
    return signal_results, best_oof_pred, best_test_pred


def sweep_catboost_probability_blend():
    """Measure whether CatBoost contributes non-redundant probability signal."""
    rows = []
    for catboost_weight in [0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
        blend_oof = normalize_rows(
            (1.0 - catboost_weight) * ensemble_oof_proba
            + catboost_weight * catboost_oof_proba
        )
        blend_test = normalize_rows(
            (1.0 - catboost_weight) * ensemble_test_proba
            + catboost_weight * catboost_test_proba
        )
        pred = proba_to_labels(blend_oof)
        rows.append(
            {
                'catboost_weight': catboost_weight,
                'lgbm_xgb_weight': 1.0 - catboost_weight,
                'accuracy': accuracy_score(y, pred),
                'balanced_accuracy': balanced_accuracy_score(y, pred),
                'macro_f1': f1_score(y, pred, average='macro'),
                'weighted_f1': f1_score(y, pred, average='weighted'),
            }
        )

    blend_results = pd.DataFrame(rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)
    best_weight = float(blend_results.loc[0, 'catboost_weight'])
    best_oof_proba = normalize_rows(
        (1.0 - best_weight) * ensemble_oof_proba + best_weight * catboost_oof_proba
    )
    best_test_proba = normalize_rows(
        (1.0 - best_weight) * ensemble_test_proba + best_weight * catboost_test_proba
    )
    return (
        blend_results,
        proba_to_labels(best_oof_proba),
        proba_to_labels(best_test_proba),
        best_oof_proba,
        best_test_proba,
        best_weight,
    )


calibration_results, calibrated_oof, calibrated_test_pred, calibrated_oof_proba, calibrated_test_proba = sweep_class_prior_calibration()

print('Top calibrated class-prior candidates:')
display(calibration_results.head(15))
calibration_results.to_csv(WORK_DIR / 'calibration_blend_results.csv', index=False)
display_diagnostics('calibrated_lgbm_xgb_domain_ensemble', calibrated_oof)


if RUN_CATBOOST_DIVERSITY and CATBOOST_AVAILABLE:
    catboost_oof, catboost_test_pred, catboost_oof_proba, catboost_test_proba = evaluate_catboost_native()
    display_diagnostics('catboost_native_balanced', catboost_oof)

    (
        catboost_blend_results,
        catboost_blend_oof,
        catboost_blend_test_pred,
        catboost_blend_oof_proba,
        catboost_blend_test_proba,
        best_catboost_weight,
    ) = sweep_catboost_probability_blend()
    print('CatBoost diversity blend candidates:')
    display(catboost_blend_results)
    catboost_blend_results.to_csv(WORK_DIR / 'catboost_blend_results.csv', index=False)
    display_diagnostics('lgbm_xgb_catboost_domain_blend', catboost_blend_oof)

    signal_results, signal_oof, signal_test_pred = sweep_catboost_signal_engine()
    print('Top CatBoost signal-engine candidates:')
    display(signal_results.head(15))
    signal_results.to_csv(WORK_DIR / 'catboost_signal_results.csv', index=False)
    display_diagnostics('catboost_signal_engine', signal_oof)
else:
    print('CatBoost diversity experiment disabled or unavailable.')
    catboost_oof = ensemble_oof
    catboost_test_pred = ensemble_test_pred
    catboost_oof_proba = ensemble_oof_proba
    catboost_test_proba = ensemble_test_proba
    catboost_blend_results = pd.DataFrame()
    catboost_blend_oof = ensemble_oof
    catboost_blend_test_pred = ensemble_test_pred
    catboost_blend_oof_proba = ensemble_oof_proba
    catboost_blend_test_proba = ensemble_test_proba
    best_catboost_weight = 0.0
    signal_results = pd.DataFrame([{'notes': 'skipped'}])
    signal_oof = ensemble_oof
    signal_test_pred = ensemble_test_pred
    pd.DataFrame().to_csv(WORK_DIR / 'catboost_blend_results.csv', index=False)
    pd.DataFrame().to_csv(WORK_DIR / 'catboost_signal_results.csv', index=False)


def build_stack_features(component_bank: dict) -> np.ndarray:
    """Concatenate base-model probability columns into meta-features."""
    return np.hstack([normalize_rows(component_bank[name]) for name in sorted(component_bank)])


def evaluate_oof_stacking(oof_components: dict, test_components: dict, label: str = 'oof_stack'):
    """Train a multinomial logistic meta-learner on OOF probabilities."""
    X_meta_oof = build_stack_features(oof_components)
    X_meta_test = build_stack_features(test_components)
    meta_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 11)

    sweep_rows = []
    best = None
    for C in [0.25, 0.5, 1.0, 2.0, 4.0]:
        for class_weight in ['balanced', None]:
            oof_proba = np.zeros((len(X_meta_oof), len(classes)), dtype='float64')
            for fold, (trn_idx, val_idx) in enumerate(meta_cv.split(X_meta_oof, y), 1):
                meta = LogisticRegression(
                    multi_class='multinomial',
                    solver='lbfgs',
                    C=C,
                    class_weight=class_weight,
                    max_iter=2000,
                    random_state=SEED,
                )
                meta.fit(X_meta_oof[trn_idx], y_enc[trn_idx])
                oof_proba[val_idx] = meta.predict_proba(X_meta_oof[val_idx])

            oof_proba = normalize_rows(oof_proba)
            oof_pred = proba_to_labels(oof_proba)
            row = {
                'C': C,
                'class_weight': 'balanced' if class_weight == 'balanced' else 'none',
                'accuracy': accuracy_score(y, oof_pred),
                'balanced_accuracy': balanced_accuracy_score(y, oof_pred),
                'macro_f1': f1_score(y, oof_pred, average='macro'),
                'weighted_f1': f1_score(y, oof_pred, average='weighted'),
                'oof_proba': oof_proba,
            }
            sweep_rows.append(row)
            print(
                f'{label} C={C} weight={row["class_weight"]}: '
                f'bal_acc={row["balanced_accuracy"]:.5f}, macro_f1={row["macro_f1"]:.5f}'
            )

    stack_results = pd.DataFrame(sweep_rows).sort_values(
        ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
    ).reset_index(drop=True)
    best = stack_results.iloc[0]
    best_oof_proba = normalize_rows(best['oof_proba'])
    best_meta = LogisticRegression(
        multi_class='multinomial',
        solver='lbfgs',
        C=float(best['C']),
        class_weight=('balanced' if best['class_weight'] == 'balanced' else None),
        max_iter=2000,
        random_state=SEED,
    )
    best_meta.fit(X_meta_oof, y_enc)
    best_test_proba = normalize_rows(best_meta.predict_proba(X_meta_test))
    stack_save = stack_results.drop(columns=['oof_proba'])
    return (
        proba_to_labels(best_oof_proba),
        proba_to_labels(best_test_proba),
        best_oof_proba,
        best_test_proba,
        stack_save,
        float(best['C']),
        str(best['class_weight']),
    )


if RUN_STACKING:
    stack_oof_components = {
        'lgbm': domain_oof_lgb if domain_oof_lgb is not None else ensemble_oof_proba,
        'xgb': domain_oof_xgb,
        'hgb': hgb_oof_proba,
    }
    stack_test_components = {
        'lgbm': domain_test_lgb if domain_test_lgb is not None else ensemble_test_proba,
        'xgb': domain_test_xgb,
        'hgb': hgb_test_proba,
    }
    if RUN_CATBOOST_DIVERSITY and CATBOOST_AVAILABLE:
        stack_oof_components['catboost'] = catboost_oof_proba
        stack_test_components['catboost'] = catboost_test_proba

    (
        stack_oof,
        stack_test_pred,
        stack_oof_proba,
        stack_test_proba,
        stack_results,
        best_stack_C,
        best_stack_class_weight,
    ) = evaluate_oof_stacking(stack_oof_components, stack_test_components)

    print('OOF stacking sweep:')
    display(stack_results)
    print(
        'Best stack config:',
        {
            'components': sorted(stack_oof_components),
            'C': best_stack_C,
            'class_weight': best_stack_class_weight,
            'balanced_accuracy': float(stack_results.loc[0, 'balanced_accuracy']),
            'macro_f1': float(stack_results.loc[0, 'macro_f1']),
        },
    )
    display_diagnostics('oof_probability_stack', stack_oof)
    stack_results.to_csv(WORK_DIR / 'stack_results.csv', index=False)
else:
    print('OOF stacking disabled.')
    stack_oof = ensemble_oof
    stack_test_pred = ensemble_test_pred
    stack_oof_proba = ensemble_oof_proba
    stack_test_proba = ensemble_test_proba
    best_stack_C = None
    best_stack_class_weight = None
    stack_results = pd.DataFrame()
    stack_oof_components = {}
    pd.DataFrame().to_csv(WORK_DIR / 'stack_results.csv', index=False)


results = [
    metric_row(
        'hgb_balanced_domain',
        hgb_oof,
        hgb_test_pred,
        notes='Continuity anchor with balanced sample weights and domain features.',
    ),
    metric_row(
        'lgbm_xgb_domain_ensemble',
        ensemble_oof,
        ensemble_test_pred,
        notes=f'Balanced LGBM/XGB probability blend selected by OOF sweep; LGBM weight={best_lgb_weight:.2f}.',
    ),
    metric_row(
        'lgbm_xgb_tuned_domain_ensemble',
        tuned_oof,
        tuned_test_pred,
        notes=(
            f'Tuned LGBM/XGB domain blend; lgbm={best_tuned_lgbm}; xgb={best_tuned_xgb}; '
            f'LGBM weight={best_tuned_lgb_weight:.2f}; GPU={GPU_AVAILABLE}.'
        ),
    ),
    metric_row(
        'lgbm_xgb_multiseed_domain_ensemble',
        multiseed_oof,
        multiseed_test_pred,
        notes=(
            f'Multi-seed averaged LGBM/XGB domain blend; seeds={MULTI_SEEDS}; '
            f'LGBM weight={best_multiseed_lgb_weight:.2f}; enabled={RUN_MULTI_SEED}.'
        ),
    ),
    metric_row(
        'lgbm_xgb_fivefold_domain_ensemble',
        fivefold_oof,
        fivefold_test_pred,
        notes=(
            f'Five-fold v8 LGBM/XGB domain blend; '
            f'LGBM weight={best_fivefold_lgb_weight:.2f}; enabled={RUN_FIVE_FOLD}.'
        ),
    ),
    metric_row(
        'crossfit_threshold_lgbm_xgb',
        crossfit_threshold_oof,
        crossfit_threshold_test_pred,
        notes=(
            f'Five-fold cross-fitted class multipliers; test fit={crossfit_test_fit_multiplier:.2f}; '
            f'test unhealthy={crossfit_test_unhealthy_multiplier:.2f}; '
            f'enabled={RUN_CROSSFIT_THRESHOLDS}.'
        ),
    ),
    metric_row(
        'calibrated_lgbm_xgb_domain_ensemble',
        calibrated_oof,
        calibrated_test_pred,
        notes=str(calibration_results.loc[0, 'notes']),
    ),
    metric_row(
        'lgbm_xgb_geometry_ensemble',
        geometry_oof,
        geometry_test_pred,
        notes=(
            f'Domain + geometry forge LGBM/XGB blend; extra cols={len(GEOMETRY_EXTRA_COLS)}; '
            f'LGBM weight={best_geometry_lgb_weight:.2f}; enabled={RUN_GEOMETRY_FORGE}.'
        ),
    ),
    metric_row(
        'oof_probability_stack',
        stack_oof,
        stack_test_pred,
        notes=(
            f'OOF stack over {sorted(stack_oof_components) if stack_oof_components else []}; '
            f'C={best_stack_C}; class_weight={best_stack_class_weight}; enabled={RUN_STACKING}.'
        ),
    ),
]

candidate_predictions = {
    'hgb_balanced_domain': (hgb_oof, hgb_test_pred),
    'lgbm_xgb_domain_ensemble': (ensemble_oof, ensemble_test_pred),
    'lgbm_xgb_tuned_domain_ensemble': (tuned_oof, tuned_test_pred),
    'lgbm_xgb_multiseed_domain_ensemble': (multiseed_oof, multiseed_test_pred),
    'lgbm_xgb_fivefold_domain_ensemble': (fivefold_oof, fivefold_test_pred),
    'crossfit_threshold_lgbm_xgb': (crossfit_threshold_oof, crossfit_threshold_test_pred),
    'calibrated_lgbm_xgb_domain_ensemble': (calibrated_oof, calibrated_test_pred),
    'lgbm_xgb_geometry_ensemble': (geometry_oof, geometry_test_pred),
    'oof_probability_stack': (stack_oof, stack_test_pred),
}

if RUN_EXPLORATORY_ABLATIONS:
    results.extend(
        [
            metric_row(
                'hgb_lgbm_xgb_domain_blend',
                hgb_ensemble_oof,
                hgb_ensemble_test_pred,
                notes=(
                    f'HGB + LGBM/XGB probability blend selected by OOF sweep; '
                    f'HGB weight={best_hgb_weight:.2f}; ensemble weight={1.0 - best_hgb_weight:.2f}.'
                ),
            ),
            metric_row(
                'lgbm_xgb_interaction_ensemble',
                interaction_oof,
                interaction_test_pred,
                notes=(
                    f'Domain features + targeted interactions; LGBM/XGB blend; '
                    f'LGBM weight={best_interaction_lgb_weight:.2f}; '
                    f'new features={len(INTERACTION_EXTRA_COLS)}.'
                ),
            ),
        ]
    )
    candidate_predictions['hgb_lgbm_xgb_domain_blend'] = (hgb_ensemble_oof, hgb_ensemble_test_pred)
    candidate_predictions['lgbm_xgb_interaction_ensemble'] = (interaction_oof, interaction_test_pred)

if RUN_CATBOOST_DIVERSITY and CATBOOST_AVAILABLE:
    results.extend(
        [
            metric_row(
                'catboost_native_balanced',
                catboost_oof,
                catboost_test_pred,
                notes='Native-categorical CatBoost with balanced class weights.',
            ),
            metric_row(
                'lgbm_xgb_catboost_domain_blend',
                catboost_blend_oof,
                catboost_blend_test_pred,
                notes=(
                    f'LGBM/XGB + CatBoost probability blend; '
                    f'CatBoost weight={best_catboost_weight:.2f}.'
                ),
            ),
            metric_row(
                'catboost_signal_engine',
                signal_oof,
                signal_test_pred,
                notes=str(signal_results.loc[0, 'notes']),
            ),
        ]
    )
    candidate_predictions['catboost_native_balanced'] = (catboost_oof, catboost_test_pred)
    candidate_predictions['lgbm_xgb_catboost_domain_blend'] = (
        catboost_blend_oof,
        catboost_blend_test_pred,
    )
    candidate_predictions['catboost_signal_engine'] = (signal_oof, signal_test_pred)

comparison = pd.DataFrame(results).sort_values(
    ['balanced_accuracy', 'macro_f1', 'accuracy'], ascending=False
).reset_index(drop=True)

BASE_CANDIDATE = 'lgbm_xgb_domain_ensemble'
MIN_BALANCED_ACCURACY_GAIN = 0.0002
base_row = comparison.loc[comparison['candidate'] == BASE_CANDIDATE].iloc[0]
comparison['balanced_accuracy_gain_vs_base'] = comparison['balanced_accuracy'] - float(base_row['balanced_accuracy'])
comparison['macro_f1_gain_vs_base'] = comparison['macro_f1'] - float(base_row['macro_f1'])
comparison['passes_champion_gate'] = (
    (comparison['candidate'] == BASE_CANDIDATE)
    | (
        (comparison['balanced_accuracy_gain_vs_base'] >= MIN_BALANCED_ACCURACY_GAIN)
        & (comparison['macro_f1_gain_vs_base'] >= 0.0)
    )
)

eligible = comparison[comparison['passes_champion_gate']].copy()
champion_name = eligible.iloc[0]['candidate']
champion_oof, champion_test_pred = candidate_predictions[champion_name]

print('Champion:', champion_name)
print('Champion gate: require >= 0.0002 balanced-accuracy gain and no macro-F1 loss versus v8 LGBM/XGB, unless using the v8 base itself.')
display(comparison)

comparison.to_csv(WORK_DIR / 'baseline_model_comparison.csv', index=False)
blend_results.to_csv(WORK_DIR / 'blend_weight_results.csv', index=False)
pd.DataFrame({'id': train[ID_COL], 'target': y, 'oof_pred': champion_oof}).to_csv(
    WORK_DIR / 'champion_oof_predictions.csv', index=False
)


## 9. Champion Diagnostics

Before leaderboard submission, inspect whether the champion keeps a reasonable **prediction mix** and does not collapse minority classes.

In [ ]:
print(classification_report(y, champion_oof, digits=4))

cm = pd.DataFrame(confusion_matrix(y, champion_oof, labels=classes), index=classes, columns=classes)
display(cm)

mix = pd.DataFrame(
    {
        'train_target_share': y.value_counts(normalize=True),
        'champion_oof_share': pd.Series(champion_oof).value_counts(normalize=True),
        'champion_test_share': pd.Series(champion_test_pred).value_counts(normalize=True),
    }
).reindex(classes).fillna(0.0)

display(mix.style.format('{:.2%}'))

## 10. Create Notebook-Generated Submission

The submitted file should come from this public notebook output. This keeps the leaderboard record reproducible and easy to audit.

In [ ]:
submission = sample_submission.copy()
submission[TARGET] = champion_test_pred
submission_path = WORK_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)

print('Wrote:', submission_path)
display(submission.head())
display(submission[TARGET].value_counts(normalize=True).rename('submission_share').to_frame().style.format({'submission_share': '{:.2%}'}))

## 11. Submission Gate

Submit the notebook output only if these checks pass:

- `submission.csv` exists in `/kaggle/working`;
- selected champion is supported by **balanced accuracy** and **macro F1**, not only raw accuracy;
- `fit` and `unhealthy` predictions remain visible in the test mix;
- `baseline_model_comparison.csv` and `geometry_blend_results.csv` explain the v19 decision.


In [ ]:
assert submission_path.exists(), 'submission.csv was not created.'
assert len(submission) == len(test), 'Submission row count mismatch.'
assert submission[ID_COL].equals(sample_submission[ID_COL]), 'Submission IDs do not match sample submission.'
assert submission[TARGET].isna().sum() == 0, 'Submission contains missing predictions.'
assert set(submission[TARGET]).issubset(set(classes)), 'Submission contains unknown labels.'

print('Submission gate passed.')
print('Champion:', champion_name)
print('Public leaderboard anchor to beat: 0.94959')